In [1]:
import os
import re
import csv
import pandas as pd

# 1. QUESTÃO A)

Implemente um módulo capaz de: ler todos os
arquivos fornecidos; contabilizar o número de linhas de cada arquivo; identificar o tipo
geral de conteúdo de cada arquivo (texto livre, log, chat ou CSV); apresentar uma
pequena amostra de conteúdo de cada arquivo.

In [7]:
import re
import pandas as pd


""" Implemente um módulo capaz de: """
class FileReader:
    def __init__(self, filepath):
        self._filepath = filepath
        self.df = self._convert_to_df()

    """
    ler todos os arquivos fornecidos;
    """
    def get_file_name(self) -> str:
        return self._filepath

    def _convert_to_df(self) -> pd.DataFrame:
        file_name_pattern = re.compile(r".+\.([a-z]+)")
        file_type = re.match(file_name_pattern, self._filepath).group(1)
        match file_type:
            case "txt":
                return pd.read_table(self._filepath, header=None)
            case "log":
                return pd.read_table(self._filepath, header=None)
            case "csv":
                return pd.read_csv(self._filepath)
            case _:
                raise TypeError('Cannot guess this file type!')

    def check_content_type(self) -> str:
        """
        identificar o tipo geral de conteúdo de cada arquivo\n
        (texto livre, log, chat ou CSV);
        """

        # padrão rápido para validação
        quick_datetime_pattern = re.compile(
            r"(0[1-9]|(1|2)[0-9]|3(0|1))/" # dia
            r"(0[1-9]|1[0-2])/" # mẽs
            r"(199[0-9]|20[0-9][0-9])" # ano
            r"\s+" # espaço
            r"(0[0-9]|1[0-9]|2[0-3]):" # hora
            r"([0-5][0-9]):" # minuto
            r"([0-5][0-9])" # segundo
        )
        log_pattern = re.compile(fr"{quick_datetime_pattern.pattern}\s+(\[(INFO|DEBUG|ERROR|WARN)\])+")
        chat_pattern = re.compile(fr"\[{quick_datetime_pattern.pattern}\]\s+(A|Z)*")
        csv_pattern = re.compile(r"\d+;")

        def detect_type(file_content: str):
            if log_pattern.search(file_content):
                return "log"
            elif chat_pattern.search(file_content):
                return "chat"
            elif csv_pattern.search(file_content):
                return "csv"
            else:
                return "free text"

        file_content_type = detect_type(self.df.iloc[0, 0])
        return f"This is a {file_content_type} file"

    def count_lines(self) -> int:
        """ contabilizar o número de linhas de cada arquivo; """
        return self.df.shape[0]

    def show_sample(self) -> str:
        """ apresentar uma pequena amostra de conteúdo de cada arquivo. """
        return self.df.head()

    def __str__(self):
        return str(self.show_sample())


## A) Testando a classe FileReader

In [3]:
filepath = "1.2_assets/"
files_names = ["01_atendimentos_bagunçados.txt", "02_logs_mistos.log", "03_mensagens_chat.txt", "04_exportacao_suja.csv"]

fileReaders = []
for name in files_names:
    print(f"Handling file: {filepath+name}\n")
    file = FileReader(filepath=filepath+name)
    print(f"Content type: {file.check_content_type()}")
    print(f"Number of lines: {file.count_lines()}")
    print(f"Sample (head):\n{file.show_sample()}\n")
    fileReaders.append(file)


Handling file: 1.2_assets/01_atendimentos_bagunçados.txt

Content type: This is a free text file
Number of lines: 4575
Sample (head):
                                                   0
0  CHAMADO 00000 >>> falhou id erro contato de Da...
1  CHAMADO 00001 >>> [INFO] session=RcoreogrNwwm ...
2  CHAMADO 00002 >>> LIMPAR duplicado falhou id c...
3  complemento: [INFO] session=FgM0Fmxk754L user=...
4  CHAMADO 00003 >>> 16/01/2025 15:37:21 | acesso...

Handling file: 1.2_assets/02_logs_mistos.log

Content type: This is a log file
Number of lines: 8000
Sample (head):
                                                   0
0  07/09/2021 20:37:27 [INFO] service=appmobile s...
1  27/06/2022 09:48:26 [DEBUG] service=financeiro...
2  05/09/2025 04:45:34 [DEBUG] service=financeiro...
3  17/05/2021 13:42:23 [DEBUG] service=financeiro...
4  01/09/2022 03:34:42 [ERROR] service=integracao...

Handling file: 1.2_assets/03_mensagens_chat.txt

Content type: This is a chat file
Number of lines: 3239
Sample 

# 1. QUESTÃO B)

Implemente expressões regulares para identificar e extrair, no mínimo, os seguintes padrões:
    - e-mails,
    - telefones,
    - CPFs,
    - datas,
    - horários,
    - datas e horários combinados,
    - URLs,
    - valores monetários em reais,
    - nomes próprios (definir critérios e justificar).

Para cada tipo de padrão, o sistema deve:
    - aplicar a expressão regular aos arquivos;
    - extrair todas as ocorrências encontradas;
    - indicar em qual arquivo ocorreram.

## CPF Validator

In [8]:
# Créditos: https://github.com/rafahlobo/cpfValidator
def isCpfValid(cpf):
    """ If cpf in the Brazilian format is valid, it returns True, otherwise, it returns False. """
    # Check if type is str
    if not isinstance(cpf,str):
        return False

    # Remove some unwanted characters
    cpf = re.sub("[^0-9]",'',cpf)

    # Verify if CPF number is equal
    if cpf=='00000000000' or cpf=='11111111111' or cpf=='22222222222' or cpf=='33333333333' or cpf=='44444444444' or cpf=='55555555555' or cpf=='66666666666' or cpf=='77777777777' or cpf=='88888888888' or cpf=='99999999999':
        return False

    # Checks if string has 11 characters
    if len(cpf) != 11:
        return False

    sum = 0
    weight = 10

    """ Calculating the first cpf check digit. """
    for n in range(9):
        sum = sum + int(cpf[n]) * weight

        # Decrement weight
        weight = weight - 1

    verifyingDigit = 11 -  sum % 11

    if verifyingDigit > 9 :
        firstVerifyingDigit = 0
    else:
        firstVerifyingDigit = verifyingDigit

    """ Calculating the second check digit of cpf. """
    sum = 0
    weight = 11
    for n in range(10):
        sum = sum + int(cpf[n]) * weight

        # Decrement weight
        weight = weight - 1

    verifyingDigit = 11 -  sum % 11

    if verifyingDigit > 9 :
        secondVerifyingDigit = 0
    else:
        secondVerifyingDigit = verifyingDigit

    if cpf[-2:] == "%s%s" % (firstVerifyingDigit,secondVerifyingDigit):
        return True
    return False


## REFinder

In [ ]:
def matches_to_list(matches):
    return [match.group(0) for match in matches]


class REFinder:
    def __init__(self):
        # A lista de DDDs não será mais carregada de um arquivo específico aqui.
        # A regex de telefone será mais geral.
        pass

    def extract_emails(self, text: str) -> list[str]:
        printable_chars = r"[\!\#\$\%\&\'\*\+\-\/\=\?\^\_\`\{\|\}\~]"
        letters = r"([a-z]|[A-Z])"
        digits = r"[0-9]"
        user_essential = fr"({letters}|{digits}|{printable_chars})"
        domain_essential = fr"({letters}|{digits})"
        pre_final = fr"{domain_essential}+(\-{domain_essential}+)*\."
        final = r"(org|edu|br|com)"
        user = fr"{user_essential}+(\.{user_essential})*"
        domain = fr"{pre_final}+{final}"
        email_pattern = re.compile(fr"{user}\@{domain}")

        matches = re.finditer(pattern=email_pattern, string=text)
        return matches_to_list(matches)

    def extract_phone_numbers(self, text: str) -> list[str]:
        # Regex geral para números de telefone brasileiros (com ou sem DDD, com 8 ou 9 dígitos no final)
        # Exemplos: (11) 98765-4321, 11 9876-5432, 98765-4321, 9876-5432
        phone_pattern = re.compile(
            r'\b(?:\(?\d{2}\)?\s?)?'  # Opcional: (DDD) ou DDD seguido de espaço
            r'(?:9\d{4}|\d{4})'      # 9XXXX ou XXXX
            r'[\-\s]?'               # Opcional: hífen ou espaço
            r'\d{4}\b'               # XXXX final
        )
        matches = re.finditer(pattern=phone_pattern, string=text)
        return matches_to_list(matches)

    def extract_CPFs(self, text: str) -> list[str]:
        cpf_pattern = re.compile(r"[0-9]{11}|[0-9]{3}\.[0-9]{3}\.[0-9]{3}\-[0-9]{2}")
        matches = re.finditer(pattern=cpf_pattern, string=text)
        valid_cpfs = []
        matches = matches_to_list(matches)
        for match in matches:
            if isCpfValid(match):
                valid_cpfs.append(match)
        return valid_cpfs

    def extract_dates(self, text: str) -> list[str]:
        day = r"(0[1-9]|(1|2)[0-9]|3(0|1))"
        month = r"(0[1-9]|1[0-2])"
        year = r"(199[0-9]|20[0-9]{2})" # 1990 - 2099
        date_pattern = re.compile(fr"{day}\/{month}\/{year}")

        matches = re.finditer(pattern=date_pattern, string=text)
        return matches_to_list(matches)

    def extract_time(self, text: str) -> list[str]:
        hours = r"(0[1-9]|1[0-9]|2[0-3])"
        minutes = r"[0-5][0-9]"
        seconds = r"[0-5][0-9]"
        time_pattern = re.compile(fr"{hours}\:{minutes}\:{seconds}")

        matches = re.finditer(pattern=time_pattern, string=text)
        return matches_to_list(matches)

    def extract_datetime(self, text: str) -> list[str]:
        # Combinação de padrões de data e hora
        day = r"(0[1-9]|(1|2)[0-9]|3(0|1))"
        month = r"(0[1-9]|1[0-2])"
        year = r"(199[0-9]|20[0-9]{2})" # 1990 - 2099
        hours = r"(0[1-9]|1[0-9]|2[0-3])"
        minutes = r"[0-5][0-9]"
        seconds = r"[0-5][0-9]"
        datetime_pattern = re.compile(fr"{day}\/{month}\/{year}\s+{hours}\:{minutes}\:{seconds}")

        matches = re.finditer(pattern=datetime_pattern, string=text)
        return matches_to_list(matches)

    def extract_URLs(self, text: str) -> list[str]:
        letters = r"([a-z]|[A-Z])"
        digits = r"[0-9]"
        domain_essential = fr"({letters}|{digits})"
        pre_final = fr"{domain_essential}+(\-{domain_essential}+)*\."
        final = r"(org|edu|br|com)"
        domain = fr"{pre_final}+{final}"
        url_pattern = re.compile(fr"http(s)?\:\/\/www\.{domain}")

        matches = re.finditer(pattern=url_pattern, string=text)
        return matches_to_list(matches)

    def extract_currency_values(self, text: str) -> list[str]:
        currency_pattern = re.compile(r"R\$[0-9]+\,[0-9]{2}") # Assumindo duas casas decimais

        matches = re.finditer(pattern=currency_pattern, string=text)
        return matches_to_list(matches)

    def extract_proper_nouns(self, text: str) -> list[str]:
        # Critério: Palavras que começam com letra maiúscula, seguidas de letras minúsculas (incluindo acentuadas).
        # Este é uma heurística simples e pode incluir falsos positivos (ex: início de frase, abreviações).
        # Para uma detecção mais robusta, seria necessário processamento de linguagem natural (NLP) e um dicionário de nomes.
        proper_noun_pattern = re.compile(r'\b[A-ZÀ-Ú][a-zà-úü]+\b(?:[\s\-\']\b[A-ZÀ-Ú][a-zà-úü]+\b)*') # Captura nomes compostos e com apóstrofo

        matches = re.finditer(pattern=proper_noun_pattern, string=text)
        return matches_to_list(matches)

### Critérios para Extração de Nomes Próprios

Para a extração de nomes próprios (`extract_proper_nouns`), o critério adotado é uma heurística baseada na capitalização de palavras. A expressão regular utilizada (`r'\b[A-ZÀ-Ú][a-zà-úü]+\b(?:[\s\-\']\b[A-ZÀ-Ú][a-zà-úü]+\b)*'`) busca por:

1.  Uma fronteira de palavra (`\b`).
2.  Uma letra inicial maiúscula (incluindo caracteres acentuados como À, É, Ó, etc. - `[A-ZÀ-Ú]`).
3.  Seguida por uma ou mais letras minúsculas (incluindo caracteres acentuados - `[a-zà-úü]+`).
4.  Opcionalmente, pode capturar nomes compostos ou com hífen, procurando por um espaço, hífen ou apóstrofo (`[\s\-\']`), seguido por outra fronteira de palavra e o padrão de letra maiúscula e minúscula novamente (`(?:[\s\-\']\b[A-ZÀ-Ú][a-zà-úü]+\b)*`).

**Justificativa:**

Este critério é uma abordagem simples e eficaz para identificar muitas ocorrências de nomes próprios em textos em português. No entanto, é importante notar que esta é uma **heurística** e possui limitações:

*   **Falsos Positivos**: Pode identificar como nomes próprios palavras que iniciam frases, abreviações (ex: "Dr."), ou termos técnicos capitalizados que não são nomes próprios (ex: "Sistema").
*   **Falsos Negativos**: Pode falhar em identificar nomes próprios que não seguem a capitalização padrão (ex: "maria da silva") ou que estão em contextos específicos onde a capitalização é diferente. Não lida com sobrenomes que são substantivos comuns (ex: "Silva").

Para uma detecção mais robusta e precisa de nomes próprios, seria necessário o uso de técnicas de Processamento de Linguagem Natural (NLP) mais avançadas, como Reconhecimento de Entidades Nomeadas (NER), que utilizam modelos treinados e dicionários contextuais. Para o escopo desta tarefa, a heurística baseada em capitalização serve como uma primeira aproximação razoável.

## Testando a classe REFinder

In [32]:
def reset_storage():
    """Define ou limpa o armazenamento para o estado inicial."""
    json_data_storage = {
        "data_type": [], "data_extracted": [], "origin": [],
        "classification": [], "emails": [], "phones": [],
        "cpfs": [], "dates": [], "time": [], "datetime": [],
        "urls": [], "currency": [], "proper_nouns": []
    }
    return json_data_storage


filepath_extraction = "extraction/"


# json de armazenamento de dados
json_data_storage = reset_storage()

re_finder = REFinder()

for i, f in enumerate(fileReaders):
    name = files_names[i]
    print(f"Extracting from... {name}")
    for r in f.df.itertuples():
        row = " ".join(map(str, r))
        json_data_storage["origin"].append(f.get_file_name())
        json_data_storage["emails"].extend(re_finder.extract_emails(row))
        json_data_storage["phones"].extend(re_finder.extract_phone_numbers(row))
        json_data_storage["cpfs"].extend(re_finder.extract_CPFs(row))
        json_data_storage["dates"].extend(re_finder.extract_dates(row))
        json_data_storage["time"].extend(re_finder.extract_time(row))
        json_data_storage["datetime"].extend(re_finder.extract_datetime(row))
        json_data_storage["urls"].extend(re_finder.extract_URLs(row))
        json_data_storage["currency"].extend(re_finder.extract_currency_values(row))
        json_data_storage["proper_nouns"].extend(re_finder.extract_proper_nouns(row))

    # pd.DataFrame(json_data_storage).to_csv(name + "_modified.csv", index=False)

    # Encontrar o maior comprimento entre todas as listas
    max_length = max(len(v) for v in json_data_storage.values())

    # Preencher as listas menores com None
    for key in json_data_storage:
        current_length = len(json_data_storage[key])
        if current_length < max_length:
            json_data_storage[key].extend([None] * (max_length - current_length))

    # Criar DataFrame e salvar
    new_file = filepath_extraction + name + "_extracted.csv"
    print(f"Saving as... {new_file}")
    pd.DataFrame(json_data_storage).to_csv(new_file, index=False)
    json_data_storage = reset_storage()

Extracting from... 01_atendimentos_bagunçados.txt
Saving as... extraction/01_atendimentos_bagunçados.txt_extracted.csv
Extracting from... 02_logs_mistos.log
Saving as... extraction/02_logs_mistos.log_extracted.csv
Extracting from... 03_mensagens_chat.txt
Saving as... extraction/03_mensagens_chat.txt_extracted.csv
Extracting from... 04_exportacao_suja.csv
Saving as... extraction/04_exportacao_suja.csv_extracted.csv


## Verificando cada arquivo gerado

In [34]:
new_files_names = ["01_atendimentos_bagunçados.txt_extracted.csv",
               "02_logs_mistos.log_extracted.csv",
               "03_mensagens_chat.txt_extracted.csv",
               "04_exportacao_suja.csv_extracted.csv"]

new_fileReaders = []
for name in new_files_names:
    print(f"Handling file: {filepath_extraction+name}\n")
    file = FileReader(filepath=filepath_extraction+name)
    print(f"Number of lines: {file.count_lines()}")
    print(f"Sample (head):\n{file.show_sample()}\n")
    new_fileReaders.append(file)

Handling file: extraction/01_atendimentos_bagunçados.txt_extracted.csv

Number of lines: 6122
Sample (head):
   data_type  data_extracted                                     origin  \
0        NaN             NaN  1.2_assets/01_atendimentos_bagunçados.txt   
1        NaN             NaN  1.2_assets/01_atendimentos_bagunçados.txt   
2        NaN             NaN  1.2_assets/01_atendimentos_bagunçados.txt   
3        NaN             NaN  1.2_assets/01_atendimentos_bagunçados.txt   
4        NaN             NaN  1.2_assets/01_atendimentos_bagunçados.txt   

   classification                   emails          phones            cpfs  \
0             NaN       email=bitbh@gov.br  98) 94981-8749  637.206.058-25   
1             NaN       user=xg1go@ufpa.br   92) 948250360  399.533.854-24   
2             NaN       email=@outlook.com   91) 936445607  470.429.775-64   
3             NaN  user=975lc7@hotmail.com    94 916740197  190.058.564-24   
4             NaN  email=6cjzi@outlook.com   97) 9

# 1. QUESTÃO C)

In [ ]:
class ClassificadorCadeias:
    # Máscaras pra definir e-mails, telefones, CPFs e URLs válidos
    emailValido = re.compile(r'^[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}$')

    telefoneValido = re.compile(r'^\(?\d{2}\)?\s?\d{4,5}-?\d{4}$')

    cpfValido = re.compile(r'^\d{3}\.\d{3}\.\d{3}-\d{2}$|^\d{11}$')

    urlValida = re.compile(r'^https?://[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}(/\S*)?$')

    def __init__(self):
        pass

    # Funções para classificar os dados passados como válidos ou não
    def classificacaoEmail(self, email):
        return "E-mail válido" if self.emailValido.match(email) else "E-mail inválido"

    def classificacaoTelefone(self, telefone):
        return "Telefone válido" if self.telefoneValido.match(telefone) else "Telefone inválido"

    def classificacaoCPF(self, cpf):
        return "CPF válido" if self.cpfValido.match(cpf) else "CPF inválido"

    def classificacaoURL(self, url):
        return "URL válida" if self.urlValida.match(url) else "URL inválida"

    # Função para verificar inconsistências no arquivo CSV passado
    def inconsistenciaCSV(self, linha):
        if len(linha) != 8:
            return "Inconsistência identificada: número de campos incorreto"

        for coluna in linha:
            if coluna.strip() == '':
                return "Inconsistência identificada: campo vazio"

        return "Inconsistência não identificada"

    # Função para ler os demais tipos de arquivos (ex: .txt)
    def lerArquivos(self, arquivo):
        saidas =[] # Armazenamento dos resultados

        with open(arquivo, 'r', encoding='utf-8') as f:
            arq = f.read()
            f.close()

        for nlinha, linha in enumerate(arq, 1): # Cada linha do arquivo é percorrida
            palavras = re.findall(r'\S+', linha) # Divisão da linha em palavras separadas por espaços

            # Para cada palavra 'p' em palavras, é verificado se condiz com o tipo de dado que aparenta ser
            # e 'saidas' adquire o resultado, caso seja válido
            for p in palavras:
                if '@' in p:
                    classificEmail = self.classificacaoEmail(p)
                    saidas.append({'Arquivo': arquivo,
                                    'Linha': nlinha,
                                    'Tipo': 'email',
                                    'Valor': p,
                                    'classificacao': classificEmail})

                if re.search(r'\d{4,5}[- ]?\d{4}', p):
                    classificTel = self.classificacaoTelefone(p)
                    saidas.append({'Arquivo': arquivo,
                                   'Linha': nlinha,
                                   'Tipo': 'telefone',
                                   'Valor': p,
                                   'classificacao': classificTel})

                if re.search(r'\d{3}\.\d{3}\.\d{3}-\d{2}|\d{11}', p):
                    classificCPF = self.classificacaoCPF(p)
                    saidas.append({'Arquivo': arquivo,
                                   'Linha': nlinha,
                                   'Tipo': 'CPF',
                                   'Valor': p,
                                   'classificacao': classificCPF})

                if re.search(r'https?://', p):
                    classificURL = self.classificacaoURL(p)
                    saidas.append({'Arquivo': arquivo,
                                   'Linha': nlinha,
                                   'Tipo': 'URL',
                                   'Valor': p,
                                   'classificacao': classificURL})
        return saidas


    # Função para ler arquivo acsv
    def lerCSV(self, arquivo):
        saidas = [] # Armazenamento dos resultados

        with open(arquivo, 'r', encoding='utf-8') as f:
            leitor = csv.reader(f, delimiter=';')
            f.close()

        try:
            cabecalho = next(leitor) # A primeira linha do CSV é lida como cabeçalho
        except StopIteration:
            return saidas # Se o arquivo estiver vazio, 'saidas' é retornada vazia

        # É percorrida cada linha do arquivo, exceto pelo cabeçalho
        for nlinha, linha in enumerate(leitor, 2):

            # É verificado se há inconsistências na linha
            inconsistencia = self.inconsistenciaCSV(linha)

            # Se houver inconsistência, essa etapa é ignorada por meio de 'continue'
            if inconsistencia != 'Inconsistência não identificada':
                saidas.append({'Arquivo': arquivo,
                            'Linha': nlinha,
                            'Tipo': 'CSV',
                            'Valor': ';'.join(linha),
                            'classificacao': inconsistencia})
                continue

            # Não havendo inconsistência, verifica-se cada campo da linha para validar as palavras
            for i, j in enumerate(linha):

                if '@' in j:
                    classificEmail = self.classificacaoEmail(j)
                    saidas.append({
                        'Arquivo': arquivo,
                        'Linha': nlinha,
                        'Coluna': cabecalho[i] if i < len(cabecalho) else f"col_{i+1}",
                        'Tipo': 'email',
                        'Valor': j,
                        'classificacao': classificEmail
                        })

                if re.search(r'\d{4,5}[- ]?\d{4}', j):
                    classificTel = self.classificacaoTelefone(j)
                    saidas.append({
                        'Arquivo': arquivo,
                        'Linha': nlinha,
                        'Coluna': cabecalho[i] if i < len(cabecalho) else f"col_{i}",
                        'Tipo': 'telefone',
                        'Valor': j,
                        'classificacao': classificTel
                        })

                if re.search(r'\d{3}\.\d{3}\.\d{3}-\d{2}|\d{11}', j):
                    classificCPF = self.classificacaoCPF(j)
                    saidas.append({
                        'Arquivo': arquivo,
                        'Linha': nlinha,
                        'Coluna': cabecalho[i] if i < len(cabecalho) else f"col_{i}",
                        'Tipo': 'cpf',
                        'Valor': j,
                        'classificacao': classificCPF
                    })

                if re.search(r'https?://', j):
                    classificURL = self.classificacaoURL(j)
                    saidas.append({
                        'Arquivo': arquivo,
                        'Linha': nlinha,
                        'Coluna': cabecalho[i] if i < len(cabecalho) else f"col_{i}",
                        'Tipo': 'url',
                        'Valor': j,
                        'classificacao': classificURL
                    })

        return saidas

    # Função para verificar qual função de leitura de arquivos deve ser chamada (de CSV ou de demais arquivos)
    def qualArquivo(self, arquivo):
        return self.lerCSV(arquivo) if arquivo.endswith('.csv') else self.lerArquivos(arquivo)


## Testando a classe ClassificadorCadeias

In [ ]:
classificador = ClassificadorCadeias()


# json de armazenamento de dados
json_data_storage = reset_storage()

for i, f in enumerate(fileReaders):
    name = files_names[i]
    print(f"Extracting from... {name}")
    for r in f.df.itertuples():
        row = " ".join(map(str, r))

        emails = re_finder.extract_emails(row)
        for email in emails:
            json_data_storage["data_type"].append("email")
            json_data_storage["data_extracted"].append(email)
            json_data_storage["origin"].append(f.get_file_name())
            json_data_storage["classification"].append(
                classificador.classificacaoEmail(email)
            )

        phones = re_finder.extract_phone_numbers(row)
        for phone in phones:
            json_data_storage["data_type"].append("telefone")
            json_data_storage["data_extracted"].append(phone)
            json_data_storage["origin"].append(f.get_file_name())
            json_data_storage["classification"].append(
                classificador.classificacaoTelefone(phone)
            )

        cpfs = re_finder.extract_CPFs(row)
        for cpf in cpfs:
            json_data_storage["data_type"].append("cpf")
            json_data_storage["data_extracted"].append(cpf)
            json_data_storage["origin"].append(f.get_file_name())
            json_data_storage["classification"].append(
                classificador.classificacaoCPF(cpf)
            )

        urls = re_finder.extract_URLs(row)
        for url in urls:
            json_data_storage["data_type"].append("url")
            json_data_storage["data_extracted"].append(url)
            json_data_storage["origin"].append(f.get_file_name())
            json_data_storage["classification"].append(
                classificador.classificacaoURL(url)
            )

    # Criar DataFrame e salvar
    new_file = filepath_extraction + name + "_extracted.csv"
    print(f"Saving as... {new_file}")
    pd.DataFrame(json_data_storage).to_csv(new_file, index=False)
    json_data_storage = reset_storage()

# 1. QUESTÃO D)

"Organização dos dados extraídos. Os dados extraídos devem ser estruturados, como:
dicionários, listas ou objetos; arquivos JSON ou CSV. Cada ocorrência deve conter,
sempre que possível: tipo do dado; valor extraído; arquivo de origem; classificação
(válido ou inválido)."

Como demonstrado nos testes anteriores, houveram produções de novos arquivos .csv que contém:
- tipo do dado;
- valor extraído;
- arquivo de origem;
- classificação


1. QUESTÃO E)

2. QUESTÃO